# RoboQuest 2026 — 解説・実験編
[クイックスタート編](quickstart_ja.ipynb) と同じモデルを使い、仕組みを理解しながらコードを変更します。初めてでも上から実行できます。

**目標：観測・行動・報酬の関係を説明し、設定を1つ変えて結果を比較すること。**
学習や保存の処理は両教材で共通です。クイックスタートの実験フォルダを指定すれば、再学習せずにモデルを表示・評価できます。別の実験を行うときは実験名を変えてください。


ローカルで検証した環境を再現する場合は、配布された `walk_colab_bundle.zip` をColab左側の「ファイル」にアップロードしてからセットアップを実行します。ライブラリの版と学習コードを揃えますが、MacとColabで計算結果が完全に同一になることを保証するものではありません。


新規学習は、見本の学習と約1万ステップのPPO微調整を行います。保存済みモデルを使う場合はこの学習時間を省けます。

現在の初期設定は、約1.5 Hzのゆっくりした脚運び・停止・再発進の見本を先に覚え、PPOで微調整する方式です。教師あり学習と強化学習を組み合わせています。高さを保ちながら、速すぎる関節運動と足先以外の床接触を減点します。

GitHubに同梱した検証済みモデル（ZIPをアップロードした場合はZIP内のモデル）は `retrain_walk=False` で読み込めます。設定を変えて学習するときは `True` にします。歩行学習の初期値は見本約3万ステップ＋PPO約1万ステップで、PPOの入力正規化を固定しています。学習率や追加ステップ数を大きくすると、覚えた歩き方が崩れる場合があります。後退・横移動・旋回は別に評価が必要です。

保存済み歩行モデルを見る場合は、セットアップ後に末尾の「保存したモデルを使う」へ進んでください。


In [ ]:
#@title 🔧 セットアップ（最初に一度だけ実行してください）

import subprocess, sys, os, zipfile

print(f'Python {sys.version_info.major}.{sys.version_info.minor} で実行中')

subprocess.run(
    'command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y -q ffmpeg)',
    shell=True, check=False)

# ローカル検証済みのコード一式を使う場合は、先にこのZIPをColabにアップロード。
_bundle = '/content/walk_colab_bundle.zip'
if os.path.isfile(_bundle):
    with zipfile.ZipFile(_bundle) as archive:
        for member in archive.namelist():
            destination = os.path.realpath(os.path.join('/content', member))
            if not destination.startswith('/content/RoboQuest2026/'):
                raise ValueError('想定外のファイル名を含むZIPです。')
        archive.extractall('/content')
    print('ローカルと同じコード・モデルを読み込みました。')
elif not os.path.exists('/content/RoboQuest2026/.git'):
    print('リポジトリをダウンロード中...')
    subprocess.run(['git', 'clone', '-q',
        'https://github.com/SingularityBattleQuest/RoboQuest2026.git',
        '/content/RoboQuest2026'], check=True)
else:
    print('リポジトリを最新化中...')
    subprocess.run(['git', '-C', '/content/RoboQuest2026', 'pull', '--ff-only', 'origin', 'main', '-q'],
                   check=False)

os.chdir('/content/RoboQuest2026')
if '/content/RoboQuest2026' not in sys.path:
    sys.path.insert(0, '/content/RoboQuest2026')

print('ライブラリをインストール中...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    '-r', 'requirements.txt',
], check=True)

print('ブラウザビューアー (mjswan) をインストール中...')
_mjswan_flags = ['--ignore-requires-python'] if sys.version_info >= (3, 13) else []
MJSWAN_AVAILABLE = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *_mjswan_flags, '-c', 'requirements-training.txt', 'mjswan==0.8.2'],
).returncode == 0
if not MJSWAN_AVAILABLE:
    print('⚠ mjswan のインストールに失敗しました。ビューアーのセルだけが使えません。')
    print('  学習・数値評価のセルはそのまま実行できます。講師に連絡してください。')

print('Go2 ロボットモデルをダウンロード中...')
subprocess.run([sys.executable, 'scripts/download_models.py'], check=True)

print('\n✅ セットアップ完了！次のセルへ進んでください。')


In [ ]:
#@title 📁 保存先（実験ごとに experiment_name を変える）
team_name = '自分のチーム名' #@param {type:"string"}
experiment_name = 'baseline' #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
for value in (team_name, experiment_name):
    if not value or value in ('.', '..') or '/' in value or '\\' in value:
        raise ValueError('チーム名・実験名にはフォルダ名を1つ指定してください')
SAVE_DIR = Path('/content/drive/MyDrive/RoboQuest2026') / team_name / experiment_name
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'保存先: {SAVE_DIR}')


In [ ]:
#@title 共通機能を読み込む
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
# 共通機能（学習処理は scripts/notebook_workflow.py）
from pathlib import Path
from copy import deepcopy
from roboquest.utils.reward_utils import WalkRewardConfig, FleeRewardConfig
from scripts.notebook_workflow import (FLEE_PPO, train_policy, evaluate_flee, load_bundled_walk)
from scripts.bootstrap_smooth_walk import (SMOOTH_PPO as WALK_PPO,
    SMOOTH_REWARD as WALK_FORWARD_REWARD, SMOOTH_ENV as WALK_FORWARD_ENV,
    SMOOTH_STEPS as WALK_FORWARD_STEPS, train_smooth_walk)
from scripts.export_for_web import export_named_policy, export_all_for_web


## 1. ロボットは何を学ぶ？
強化学習では、ロボットが **観測 → 行動 → 報酬** を繰り返し、累積報酬を大きくする方策（ポリシー）を学びます。ここではPPOという手法と、数値ベクトルを処理するMLPを使います。

2つの役割を分ける階層型です。

**鬼の位置や姿勢 → 逃げモデル → 速度指令 → 歩行モデル → 12関節の目標角度 → シミュレーター**

歩行モデルを先に学習し、逃げ学習中は固定します。逃げモデルは足の動かし方を一から覚えず、移動方向の選択に集中できます。入出力の次元と順序は共通です。ただし座標変換・制御周期を修正したため、旧環境のモデルをそのまま同じ条件のモデルとして扱うことはできません。

### 歩行モデル：観測45次元 → 行動12次元
| 観測の範囲 | 内容 |
|---|---|
| `[0:3]` | 前後・左右・旋回の速度指令 |
| `[3:6]` | 胴体の角速度 |
| `[6:9]` | 胴体から見た重力方向 |
| `[9:21]` | 立ち姿勢からの関節角度差 |
| `[21:33]` | 関節角速度 |
| `[33:45]` | 前回の行動 |

行動は `[-1, 1]` の12個の値です。`立ち姿勢 + 0.6 × 行動` を目標関節角度に変換します。制御周期は50Hzです。


## 2. 歩行の報酬を設計する
速度追跡の報酬は指令への近さを評価します。重みを増やすことは、速度そのものを指定することとは違います。
姿勢の傾き・トルク・行動の急変は、負の重みでペナルティにします。例えば `orientation_weight` を `-1.0` から `-2.0` にすると傾きを強く罰します。転倒ペナルティは正の大きさを設定し、環境側で引きます。

他にも足のリズムや滑りの項があります。まずは1項目だけ変えて、同じ速度指令で歩行を比較しましょう。報酬を変えても望んだ動きになる保証はなく、他の項との釣り合いが大切です。


In [ ]:
#@title 歩行報酬の設定
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
walk_cfg = WalkRewardConfig(**WALK_FORWARD_REWARD)
# 例: walk_cfg.lin_vel_weight = 4.0


### PPOの設定とネットワーク
| 設定 | 意味・変更時の考え方 |
|---|---|
| `learning_rate` | 1回の更新の大きさ。小さくすると更新が慎重になりますが、学習が遅くなる場合があります |
| `n_steps` | 各環境で更新前に集めるステップ数 |
| `batch_size` | 更新に使う小分けデータの数。`n_steps × num_envs` を割り切れる値が扱いやすいです |
| `n_epochs` | 集めたデータを繰り返し学習する回数 |
| `gamma` | 将来の報酬をどれだけ重視するか |
| `gae_lambda` | 価値推定におけるバイアスと分散のバランス |
| `ent_coef` | 行動の多様性を促す強さ |
| `net_arch` | 隠れ層のユニット数。大きくすると計算量も増えます |

`timesteps` は全環境で集める合計ステップ数で、更新単位によって指定値を超えることがあります。`num_envs` は並列の環境数ですが、この教材の実行方式では増やすだけで必ず高速化するわけではありません。
`seed` は乱数の初期値です。同じ条件で比較する助けになりますが、実行環境をまたぐ完全一致を保証しません。


In [ ]:
#@title 歩行学習の設定
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
seed = 0
walk_timesteps = WALK_FORWARD_STEPS
walk_num_envs = 4
walk_ppo = deepcopy(WALK_PPO)
walk_ppo['learning_rate'] = 1e-6
walk_ppo['policy_kwargs']['net_arch'] = [256, 256, 128]
print(walk_ppo)


In [ ]:
#@title 🚀 歩行モデルを用意する（同名ファイルを上書き）
retrain_walk = False #@param {type:"boolean"}
bundle_dir = Path('models/teams/bundled_walk')
if not bundle_dir.is_dir():
    bundle_dir = Path('models/pretrained/smooth_walk')
if not retrain_walk and bundle_dir.is_dir():
    load_bundled_walk(SAVE_DIR, bundle_dir)
else:
    import time, shutil
    smooth_dir = Path(SAVE_DIR) / f"smooth_{time.time_ns()}"
    train_smooth_walk(smooth_dir, steps=walk_timesteps, seed=seed,
        reward_config=walk_cfg, ppo_kwargs=walk_ppo, num_envs=walk_num_envs)
    for name in ("walk_model.zip", "walk_model_vecnorm.pkl", "walk_params.json",
                 "walk_smooth_curriculum.json"):
        shutil.copy2(smooth_dir / name, Path(SAVE_DIR) / name)
export_named_policy('walk', SAVE_DIR / 'walk_model', SAVE_DIR / 'walk_model_vecnorm.pkl',
                    '/content/RoboQuest2026/webapp/models', verify=True)


In [ ]:
#@title 歩行を数値で確認する（停止・前後・左右・旋回、各3回）
from scripts.tune_walk import evaluate as evaluate_walk
walk_evaluation = evaluate_walk(SAVE_DIR)
print('全項目合格' if walk_evaluation['passed'] else '未達の項目があります。evaluation.json を確認してください。')


### 歩行を確認する
前後・左右・旋回の指令をそれぞれ試し、転倒しないか、指令に従うかを見ます。まず歩行を安定させてから逃げ学習に進んでください。


In [ ]:
#@title 🎮 歩行ビューアー（mjswan — ブラウザ内 MuJoCo + 学習済みポリシー）
#@markdown 学習した Walk ポリシーがブラウザ内でリアルタイム動作します。
#@markdown パネルのスライダー（Forward / Lateral / Yaw）で速度コマンドを入力してください。
#@markdown キー操作は `c`（パネルの開閉）と `r`（リセット）のみです。
#@markdown ※ 最初のビューアー実行時はブラウザ用の表示エンジンをビルドするため1〜3分ほどかかります（2回目以降はすぐ表示されます）。

import mjswan
from scripts.build_mjswan_viewer import build_walk

app = build_walk(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_walk_dist',
)
from scripts.launch_colab_viewer import launch_colab_viewer
if 'walk_viewer_server' in globals():
    walk_viewer_server.shutdown()
    walk_viewer_server.server_close()
walk_viewer_server = launch_colab_viewer('/tmp/rq_walk_dist', height=620)


## 3. 逃げ方の報酬を設計する
逃げモデルは5Hzで速度指令3個を出し、その間に歩行モデルが10回制御します。

| 観測の範囲 | 内容 |
|---|---|
| `[0:2]` | 鬼への相対位置 dx, dy |
| `[2]` | 鬼までの距離 |
| `[3:6]` | 胴体の角速度 |
| `[6:9]` | 重力方向 |
| `[9]` | 残り時間（1から0） |

合計10次元です。行動は `[vx, vy, omega]` の3次元で、各値は `[-1, 1]` に制限されます。

5m四方のアリーナで60秒逃げ切ることが目標です。低レベルの各ステップで、生存報酬 `survival_weight / 10` と距離報酬 `distance_weight × 距離 × 0.01` を足します。捕まると `tag_penalty`、転倒すると `fall_penalty` を引きます。

`oni_speed` は鬼が低レベル1ステップに進む距離です。デフォルトの `0.025` は、50Hzでは約1.25m/sに相当します。これは報酬の重みではなく課題の難しさを変えます。


In [ ]:
#@title 逃げ方の報酬と学習設定
#@markdown 設定を直接編集するときは、セルのメニューからコードを表示してください。
flee_cfg = FleeRewardConfig(
    survival_weight=0.5,
    distance_weight=1.0,
    tag_penalty=50.0,
    fall_penalty=20.0,
)
oni_speed = 0.025
flee_timesteps = 200_000
flee_num_envs = 2
flee_ppo = deepcopy(FLEE_PPO)
flee_ppo['learning_rate'] = 1e-4
flee_ppo['policy_kwargs']['net_arch'] = [128, 128]


In [ ]:
#@title 👹 逃げ方を学習して保存（歩行モデルを固定して使用）
train_policy('flee', SAVE_DIR, flee_cfg, flee_timesteps, flee_num_envs, flee_ppo,
             seed=seed, oni_speed=oni_speed)
export_all_for_web(walk_model=SAVE_DIR / 'walk_model',
                   walk_vecnorm=SAVE_DIR / 'walk_model_vecnorm.pkl',
                   flee_model=SAVE_DIR / 'flee_model',
                   flee_vecnorm=SAVE_DIR / 'flee_model_vecnorm.pkl',
                   save_dir='/content/RoboQuest2026/webapp/models', verify=True)


## 4. 同じ条件で結果を比較する
学習と評価で観測の尺度を揃える必要があります。`VecNormalize` は学習中に平均・分散を記録し、観測を正規化します。ZIPだけではこの情報が足りないため、PKLも一緒に保存します。
評価では保存した統計を読み込み、`training=False` で更新を止め、`norm_reward=False` で報酬の正規化を止めます。歩行・逃げの両方にそれぞれの観測統計を使います。

下のセルは固定した5つの初期配置で評価します。平均生存時間、逃げ切り、捕獲、転倒、平均距離を記録します。生存時間は高レベル周期0.2秒単位です。5回は小規模な比較なので、良さそうな設定は試行数や学習seedを増やして確かめましょう。

ブラウザのアリーナは現在、歩行モデルの手動操作のみ対応しています。逃げAIの性能判断には数値評価を使ってください。


In [ ]:
#@title 📊 保存した逃げAIを評価（再学習不要）
# セットアップ・保存先・共通機能セルを実行すれば、別の日にも評価できます。
import json
results = evaluate_flee(SAVE_DIR, seeds=(100, 101, 102, 103, 104))
for row in results:
    print(row)
print(f"平均生存時間: {sum(r['survived_seconds'] for r in results) / len(results):.1f} 秒")
print(f"逃げ切り: {sum(r['escaped'] for r in results)}/{len(results)}")
(SAVE_DIR / 'evaluation.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
#@title 🎮 鬼ごっこビューアー（mjswan — ブラウザ内 MuJoCo アリーナ）
#@markdown アリーナ（壁 + 鬼ボディ）を表示し、Walk ポリシーをスライダーで手動操作します。
#@markdown ※ Flee AI ポリシーは今後実装予定。
#@markdown ※ 最初のビューアー実行時はブラウザ用の表示エンジンをビルドするため1〜3分ほどかかります（2回目以降はすぐ表示されます）。

import mjswan
from scripts.build_mjswan_viewer import build_flee

app = build_flee(
    walk_onnx_path='/content/RoboQuest2026/webapp/models/walk_policy_normalized.onnx',
    output_dir='/tmp/rq_flee_dist',
)
from scripts.launch_colab_viewer import launch_colab_viewer
if 'flee_viewer_server' in globals():
    flee_viewer_server.shutdown()
    flee_viewer_server.server_close()
flee_viewer_server = launch_colab_viewer('/tmp/rq_flee_dist', height=620)


### 実験してみよう
1. `baseline` の結果を保存する。
2. 別の実験名にし、`orientation_weight=-2.0` で歩行から学習する。
3. 同じ速度指令で姿勢と追従を比較し、同じ評価seedで逃げ成績を比較する。
4. 「仮説・変更した値・結果・考察」を記録する。

逃げ報酬だけを比較する場合は、新しい実験フォルダに基準の `walk_model.zip`、`walk_model_vecnorm.pkl`、`walk_params.json` をコピーし、歩行学習を飛ばします。同じ歩行モデル・鬼の速度・評価seed・学習量を使うことで比較しやすくなります。

| 実験名 | 変更した値 | 平均生存秒 | 逃げ切り数 | 考察 |
|---|---|---|---|---|
| baseline | なし | 実行後に記入 | 実行後に記入 | |

さらに改造する場合は、`scripts/notebook_workflow.py` の `train_policy` が環境作成、PPO学習、保存を担当しています。環境内の報酬処理は `Go2WalkEnv` と `Go2TagHierarchicalEnv` にあります。まずこの教材の重み変更で比較方法を身につけてから、報酬項や観測の追加へ進んでください。


## 保存したモデルを使う
学習終了時に Google Drive に保存されます。保存するセルの再実行は新規学習で、同じ実験名のファイルを上書きします。歩行を学習し直したら、逃げモデルも学習し直してください。

- `walk_model.zip` と `walk_model_vecnorm.pkl`：歩行モデルと観測の正規化データ
- `flee_model.zip` と `flee_model_vecnorm.pkl`：逃げモデルと観測の正規化データ
- `walk_params.json` と `flee_params.json`：学習設定
- `evaluation.json`：評価結果

提出用にはこの実験フォルダをまとめて保管してください。この操作だけで大会への送信は行いません。
歩行モデルを見るだけなら「セットアップ」を実行した後、この末尾の表示欄へ進みます。`show_saved_walk` をオンにし、保存先（例：チーム名/baseline）を指定して実行してください。途中の学習セルは実行不要です。

逃げAIの数値評価は「セットアップ」「保存先」「共通機能」を実行してから、上の「保存した逃げAIを評価」を使います。両教材で保存形式は共通です。


In [ ]:
#@title 📂 保存済み歩行モデルを選んで表示（再学習不要）
#@markdown 表示する場合だけチェックしてください。通常の学習ではオフのままにします。
show_saved_walk = False #@param {type:"boolean"}
#@markdown RoboQuest2026 内の保存先を指定（例: チーム名/baseline、旧保存先は test）。
saved_team_name = "test" #@param {type:"string"}
#@markdown 実際のZIP名を指定（walkmodel.zipの場合は変更してください）。
saved_model_name = "walk_model.zip" #@param {type:"string"}
#@markdown 正規化ファイルが別名ならフルパスを指定。通常は空欄。
saved_vecnorm_path = "" #@param {type:"string"}

if show_saved_walk:
    from google.colab import drive
    drive.mount('/content/drive')

    from pathlib import Path
    root = Path('/content/drive/MyDrive/RoboQuest2026')
    folder = (root / saved_team_name).resolve()
    if not folder.is_relative_to(root.resolve()):
        raise ValueError('RoboQuest2026内のフォルダ名を指定してください')
    if not folder.is_dir():
        available = ', '.join(p.name for p in root.iterdir() if p.is_dir()) if root.exists() else '(Drive未接続)'
        raise FileNotFoundError(f'フォルダがありません。選べるフォルダ: {available}')
    model_path = folder / saved_model_name
    # Accept the common spelling without an underscore as well.
    if not model_path.exists() and saved_model_name == 'walk_model.zip':
        model_path = folder / 'walkmodel.zip'
    if not model_path.is_file():
        raise FileNotFoundError(f'ZIPがありません。候補: {[p.name for p in folder.glob("*.zip")]}')
    print(f'読み込むモデル: {model_path}')

    from scripts.preview_saved_walk import preview_saved_walk
    from scripts.launch_colab_viewer import launch_colab_viewer

    from scripts.build_mjswan_viewer import build_walk
    onnx_path, has_stats = preview_saved_walk(model_path, saved_vecnorm_path or None)
    app = build_walk(onnx_path, onnx_path.parent / 'viewer')
    if 'saved_walk_viewer_server' in globals():
        saved_walk_viewer_server.shutdown()
        saved_walk_viewer_server.server_close()
    saved_walk_viewer_server = launch_colab_viewer(onnx_path.parent / 'viewer', height=620)
    if not has_stats:
        from IPython.display import HTML, display
        display(HTML('<p style="color:#b45309"><b>参考プレビュー：正規化データなし。学習時の動作再現ではありません。</b></p>'))
else:
    print('保存済みモデルを表示するには show_saved_walk をオンにして、このセルを実行してください。')
